# Kaggle Predict F1 Pit Stops: Baseline Modeling

This notebook builds baseline models for predicting `PitNextLap` probabilities.

EDA reference notebook: https://www.kaggle.com/code/tuannm3812/kaggle-predict-f1-pit-stops-preprocessing-and-eda

Modeling goals:

- Establish a naive baseline.
- Train simple, medium, and stronger baseline models.
- Use stratified cross-validation with probability metrics.
- Generate a valid `submission.csv`.
- Keep feature engineering consistent with the EDA notebook.


## 1. Setup


In [ ]:
import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "PitNextLap"
ID_COL = "id"

# Set to True for quick notebook iteration. Set False for final Kaggle runs.
RUN_FAST = True
FAST_SAMPLE_SIZE = 180_000


## 2. Load Data


In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train[TARGET].mean():.5f}")
train.head()


## 3. Feature Engineering

These features mirror the EDA notebook and use only row-level information available in both train and test.


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    return reduce_memory_usage(out)


train_fe = add_features(train)
test_fe = add_features(test)

if RUN_FAST and len(train_fe) > FAST_SAMPLE_SIZE:
    train_fe = (
        train_fe.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    print(f"RUN_FAST sample shape: {train_fe.shape}")

X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET].astype("int8")
X_test = test_fe.copy()

print("Features:", X.shape[1])
print("Positive rate:", y.mean())


## 4. Preprocessing Helpers


In [ ]:
def make_onehot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse=True)


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    drop_cols = [ID_COL]
    features = df.drop(columns=[c for c in drop_cols if c in df.columns])
    cat_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in features.columns if c not in cat_cols]
    return num_cols, cat_cols


def make_linear_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = get_feature_columns(df)
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop", sparse_threshold=0.3)


def make_tree_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = get_feature_columns(df)
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop")


def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }


## 5. Model Lineup

The lineup moves from simple to more expressive:

- `Dummy`: sanity check using the class prior.
- `LogisticRegression`: linear baseline with one-hot categorical encoding.
- `HistGradientBoosting`: sklearn boosted-tree baseline with ordinal categorical encoding.
- `RandomForest` / `ExtraTrees`: bagged-tree baselines, useful but can be slower.
- Optional `LightGBM`, `XGBoost`, and `CatBoost` if installed in the Kaggle image.


In [ ]:
models = []

models.append((
    "dummy_prior",
    Pipeline([
        ("preprocess", make_linear_preprocessor(X)),
        ("model", DummyClassifier(strategy="prior")),
    ]),
))

models.append((
    "logistic_regression",
    Pipeline([
        ("preprocess", make_linear_preprocessor(X)),
        ("model", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
))

models.append((
    "hist_gradient_boosting",
    Pipeline([
        ("preprocess", make_tree_preprocessor(X)),
        ("model", HistGradientBoostingClassifier(max_iter=350, learning_rate=0.05, l2_regularization=0.05, random_state=RANDOM_STATE)),
    ]),
))

if not RUN_FAST:
    models.extend([
        (
            "random_forest",
            Pipeline([
                ("preprocess", make_tree_preprocessor(X)),
                ("model", RandomForestClassifier(n_estimators=400, min_samples_leaf=20, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced_subsample")),
            ]),
        ),
        (
            "extra_trees",
            Pipeline([
                ("preprocess", make_tree_preprocessor(X)),
                ("model", ExtraTreesClassifier(n_estimators=500, min_samples_leaf=20, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced")),
            ]),
        ),
    ])

try:
    from lightgbm import LGBMClassifier
    models.append((
        "lightgbm",
        Pipeline([
            ("preprocess", make_tree_preprocessor(X)),
            ("model", LGBMClassifier(
                objective="binary",
                n_estimators=1200 if not RUN_FAST else 500,
                learning_rate=0.03,
                num_leaves=63,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )),
        ]),
    ))
except Exception as exc:
    print("LightGBM unavailable:", exc)

try:
    from xgboost import XGBClassifier
    models.append((
        "xgboost",
        Pipeline([
            ("preprocess", make_tree_preprocessor(X)),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=900 if not RUN_FAST else 400,
                learning_rate=0.035,
                max_depth=7,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=2.0,
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
    ))
except Exception as exc:
    print("XGBoost unavailable:", exc)

print("Models:", [name for name, _ in models])


## 6. Cross-Validation


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
results = []
oof_predictions = {}

def get_positive_proba(estimator, X_part):
    proba = estimator.predict_proba(X_part)
    return proba[:, 1]

for model_name, estimator in models:
    print(f"\n=== {model_name} ===")
    start = time.time()
    oof = np.zeros(len(X), dtype=np.float32)
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        fold_estimator = clone(estimator)
        fold_estimator.fit(X_train, y_train)
        valid_pred = get_positive_proba(fold_estimator, X_valid)
        oof[valid_idx] = valid_pred

        scores = evaluate_predictions(y_valid, valid_pred)
        scores.update({"model": model_name, "fold": fold})
        fold_scores.append(scores)
        print(f"fold {fold}: auc={scores['roc_auc']:.5f}, ap={scores['average_precision']:.5f}, logloss={scores['log_loss']:.5f}")

        del fold_estimator, X_train, X_valid, y_train, y_valid
        gc.collect()

    overall = evaluate_predictions(y, oof)
    overall.update({"model": model_name, "fold": "oof", "fit_seconds": time.time() - start})
    results.extend(fold_scores)
    results.append(overall)
    oof_predictions[model_name] = oof
    print(f"OOF: auc={overall['roc_auc']:.5f}, ap={overall['average_precision']:.5f}, logloss={overall['log_loss']:.5f}, seconds={overall['fit_seconds']:.1f}")

results_df = pd.DataFrame(results)
results_df.sort_values(["fold", "roc_auc"], ascending=[True, False])


## 7. Compare Baselines


In [ ]:
summary = (
    results_df[results_df["fold"].eq("oof")]
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)
summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metric_specs = [
    ("roc_auc", False, "OOF ROC AUC"),
    ("average_precision", False, "OOF Average Precision"),
    ("log_loss", True, "OOF Log Loss"),
]

plot_df = summary.copy()
for ax, (metric, ascending, title) in zip(axes, metric_specs):
    order = plot_df.sort_values(metric, ascending=ascending)["model"]
    sns.barplot(data=plot_df, y="model", x=metric, order=order, color=sns.color_palette("viridis", 8)[4], ax=ax)
    ax.set_title(title)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 8. Train Best Model and Create Submission


In [ ]:
best_model_name = summary.iloc[0]["model"]
best_estimator = dict(models)[best_model_name]

print("Best model:", best_model_name)
best_estimator.fit(X, y)
test_pred = get_positive_proba(best_estimator, X_test)
test_pred = np.clip(test_pred, 1e-6, 1 - 1e-6)

submission = sample_submission.copy()
submission[TARGET] = test_pred
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
submission.head()


## 9. Next Experiments

After these baselines, the most useful improvements are likely:

- Run with `RUN_FAST = False` for full-data validation.
- Tune LightGBM/CatBoost if available.
- Compare feature sets with and without tyre-ratio features.
- Add cross-validated target encoding for `Driver` and `Race`.
- Review false positives/false negatives by `Compound`, `Stint`, and `RaceProgress`.
- Calibrate probabilities if log loss is the main metric.
